In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import soundfile as sf
import librosa
import numpy as np
import IPython.display as ipd
import tqdm
from torch.utils.data import Dataset, DataLoader
# Define the device
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS device")
else:
    device = torch.device("cpu")
    print("MPS not available, using CPU")

In [ ]:
class AudioResidualLSTM(nn.Module):
    def __init__(self, input_channels=1, hidden_dim=64, num_layers=2, kernel_size=3):
        """
        Args:
            input_channels: 1 for mono, 2 for stereo.
            hidden_dim: The 'memory' size of the LSTM.
            num_layers: How many LSTM layers to stack.
            kernel_size: Used for a pre-processing convolution to help the LSTM.
        """
        super(AudioResidualLSTM, self).__init__()

        self.input_channels = input_channels
        self.hidden_dim = hidden_dim

        # 1. Pre-processing Layer: A small 1D Convolution.
        # This helps the model look at a tiny window of samples
        # rather than just a single isolated point.
        self.pre_conv = nn.Conv1d(
            in_channels=input_channels,
            out_channels=hidden_dim,
            kernel_size=kernel_size,
            padding=kernel_size // 2
        )

        # 2. The Temporal Engine: LSTM
        # batch_first=True means input shape is (Batch, Sequence, Features)
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        # 3. The Projection Head: The FC/Linear Layer
        # This maps the high-dimensional LSTM state back to audio amplitude.
        self.fc = nn.Linear(hidden_dim, input_channels)

    def forward(self, x):
        """
        x: Input tensor of shape (Batch, Sequence, Channels)
           Example: (32, 1024, 1) -> 32 chunks of 1024 samples each.
        """
        # Store the original input for the residual connection
        identity = x

        # --- Step 1: Pre-processing ---
        # Conv1d expects (Batch, Channels, Sequence), so we permute
        x_conv = x.permute(0, 2, 1)
        x_conv = self.pre_conv(x_conv)

        # Bring it back to (Batch, Sequence, Hidden_Dim) for the LSTM
        x_features = x_conv.permute(0, 2, 1)

        # --- Step 2: Temporal Modeling ---
        # lstm_out shape: (Batch, Sequence, Hidden_Dim)
        lstm_out, (h_n, c_n) = self.lstm(x_features)

        # --- Step 3: Projection ---
        # Map hidden features back to audio channels
        # residual shape: (Batch, Sequence, Channels)
        residual = self.fc(lstm_out)

        # --- Step 4: The Residual Connection (CRITICAL) ---
        # We add the learned distortion back to the original clean signal.
        # Output = Clean_Signal + Learned_Distortion
        output = identity + residual

        return output

In [ ]:
class AudioAmpDataset(Dataset):
    def __init__(self, clean_path, target_path ,seq_len=512, calculate_norm=True):
        """
        Args:
            clean_path: Path to the input (DI) audio file.
            target_path: Path to the target (Distorted) audio file.
            seq_len: The number of samples per chunk.
        """

        self.clean_path = clean_path
        self.target_path = target_path
        self.seq_len = seq_len

        # Open files just to get their total lengths (metadata), not to read the whole file into RAM
        clean_info = sf.info(clean_path)
        target_info = sf.info(target_path)

        # Ensure both files are the same length
        assert clean_info.frames == target_info.frames, "Clean and Target files must be the same length."

        self.total_frames = clean_info.frames

        self.clean_max = 1.0
        self.target_max = 1.0
        if calculate_norm:
            print("Calculating global maximums for normalization (this might take a few seconds)...")
            self._compute_global_max()

        # Calculate how many valid chunks we can extract
        self.num_chunks = self.total_frames // self.seq_len

    def __len__(self):
        return self.num_chunks

    def _compute_global_max(self, block_size=44100*5):
        # This is a helper function to compute the global max amplitude for normalization.

        current_Train_max = 0.0
        current_Valid_max = 0.0
        total_block = self.total_frames // block_size + 1

        for block_i in range(total_block):
            start_frame = block_i * block_size
            frames_to_read = min(block_size, self.total_frames - start_frame)
            Train_block, _ = sf.read(self.clean_path, frames=frames_to_read, start=start_frame, dtype='float32')
            Valid_block, _ = sf.read(self.target_path, frames=frames_to_read, start=start_frame, dtype='float32')
            Block_Train_max =  np.max(np.abs(Train_block))
            Block_Valid_max =  np.max(np.abs(Valid_block))

            if Block_Train_max > current_Train_max:
                current_Train_max = Block_Train_max
            if Block_Valid_max > current_Valid_max:
                current_Valid_max = Block_Valid_max

        self.clean_max = max(current_Train_max, 1e-8)
        self.target_max = max(current_Valid_max, 1e-8)
        print(f"Global Clean Max: {self.clean_max:.4f} | Global Target Max: {self.target_max:.4f}")

        return 1.0


    def __getitem__(self, idx):
        """
        This is where 'lazy loading' happens.
        It is only called when the DataLoader requests batch 'idx'.
        """
        # Calculate the exact sample index to start reading from
        start_frame = idx * self.seq_len

        # Read exactly 'seq_len' frames from disk
        clean_chunk, _ = sf.read(self.clean_path, frames=self.seq_len, start=start_frame, dtype='float32')
        target_chunk, _ = sf.read(self.target_path, frames=self.seq_len, start=start_frame, dtype='float32')

        # If stereo, convert to mono. soundfile returns shape (frames, channels)
        if clean_chunk.ndim > 1:
            clean_chunk = clean_chunk.mean(axis=1)
        if target_chunk.ndim > 1:
            target_chunk = target_chunk.mean(axis=1)

        # Optional: Normalize the chunks right here
        # Note: True lazy normalization scales by the chunk's max. If you want global normalization,
        # you need to pre-compute the global max and divide by it here.
        clean_chunk = clean_chunk / self.clean_max
        target_chunk = target_chunk / self.target_max
        # Reshape to expected (Sequence, Features) which is (512, 1)
        clean_tensor = torch.tensor(clean_chunk, dtype=torch.float32).unsqueeze(-1)
        target_tensor = torch.tensor(target_chunk, dtype=torch.float32).unsqueeze(-1)

        return clean_tensor, target_tensor


In [ ]:
dataset = AudioAmpDataset(
    clean_path="Data/ML Guitar_ Initial_Data.wav",
    target_path="Data/ML Guitar Target Data.wav",
    seq_len=512
)
train_loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,       # Shuffle the chunks for better training
    num_workers=0,      # Uses 2 background processes to read from disk concurrently
    pin_memory=True    # Speeds up transferring data to MPS/GPU
)

In [ ]:
def train_model_with_loader(model, dataloader, epochs=5, device="mps"):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    model.to(device)
    model.train()

    for epoch in tqdm.tqdm(range(epochs), desc="Training Epochs"):
        epoch_loss = 0.0

        # The dataloader handles batching and lazy-loading from disk
        for batch_idx, (clean_batch, target_batch) in enumerate(dataloader):

            # Move the small batch to device
            clean_batch = clean_batch.to(device)
            target_batch = target_batch.to(device)

            optimizer.zero_grad()
            predictions = model(clean_batch)
            loss = criterion(predictions, target_batch)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(dataloader)
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.6f}")

# Run it
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = AudioResidualLSTM(input_channels=1, hidden_dim=128)
train_model_with_loader(model, train_loader, epochs=5, device=device)

In [ ]:
def process_new_audio(model, input_file, output_file, device, seq_len=512, batch_size=32):
    """
    Passes a raw audio file through the trained LSTM and saves the result.
    """
    model.eval() # 1. Set model to inference mode (disables dropout, etc.)

    # ==========================================
    # STEP 1: LOAD THE AUDIO
    # ==========================================
    # We use librosa here; it will force mono if stereo.
    # sr=None preserves the original sample rate.
    print(f"Loading {input_file}...")
    audio, sr = librosa.load(input_file, sr=None, mono=True)

    # Optional but recommended if you trained on normalized data
    audio = librosa.util.normalize(audio)

    # ==========================================
    # STEP 2: CHUNK THE AUDIO
    # ==========================================
    # Calculate padding so the last chunk is exactly seq_len
    remainder = len(audio) % seq_len
    if remainder != 0:
        padding = seq_len - remainder
        audio = np.pad(audio, (0, padding), mode='constant')

    # Reshape the 1D array into shape (Total_Chunks, Sequence_Length, 1)
    num_chunks = len(audio) // seq_len
    # (Chunks, Time_Steps, Features/Channels)
    chunks = audio.reshape(num_chunks, seq_len, 1)

    # Convert to PyTorch Tensor
    tensor_chunks = torch.tensor(chunks, dtype=torch.float32)

    # ==========================================
    # STEP 3: PREDICT IN BATCHES
    # ==========================================
    processed_chunks = []

    print(f"Processing {num_chunks} chunks...")
    with torch.no_grad(): # Disable gradients for speed and memory savings
        for i in range(0, num_chunks, batch_size):
            # Get a batch and move to GPU (MPS)
            batch = tensor_chunks[i : i + batch_size].to(device)

            # Pass through the model
            predictions = model(batch)

            # Move back to CPU and convert to numpy array
            processed_chunks.append(predictions.cpu().numpy())

    # ==========================================
    # STEP 4: STITCH AND SAVE
    # ==========================================
    # Concatenate all batches back into a single array
    # Shape goes from list of (Batch, 512, 1) -> (Total_Chunks, 512, 1)
    output_audio = np.concatenate(processed_chunks, axis=0)

    # Flatten back to a continuous 1D array
    output_audio = output_audio.flatten()

    # If we padded earlier, remove the padding at the very end
    if remainder != 0:
        output_audio = output_audio[:-padding]

    # Save the result
    print(f"Saving resulting audio to {output_file}...")
    sf.write(output_file, output_audio, sr)
    print("Done!")

    return output_audio


In [ ]:
# Setup device (MPS for your Mac)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Move your trained model to the device
my_trained_model = model.to(device)

# Process a new riff
result = process_new_audio(
    model=my_trained_model,
    input_file="Data/ML Guitar_ Initial_Data.wav",
    output_file="Data/My_Distorted_Riff.wav",
    device=device,
    seq_len=512,    # MUST match what you fed the model during training
    batch_size=32   # Can be anything, higher is faster but uses more memory
)

In [ ]:
ipd.Audio(result, rate=44100)